In [1]:
from pathlib import Path
import os
import sys


PROJECT_ROOT = Path.cwd().parent 
sys.path.append(str(PROJECT_ROOT))

In [2]:
import mediapipe as mp
try:
    print("Version:", mp.__version__)
    print("Solutions available:", dir(mp.solutions))
    print("OK! Đã fix thành công.")
except Exception as e:
    print("Vẫn lỗi:", e)

Version: 0.10.9
Solutions available: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'download_utils', 'drawing_styles', 'drawing_utils', 'face_detection', 'face_mesh', 'face_mesh_connections', 'hands', 'hands_connections', 'holistic', 'mediapipe', 'objectron', 'pose', 'pose_connections', 'selfie_segmentation']
OK! Đã fix thành công.


## Whisper Tokenizer

In [ ]:
from src.data.tokenizers.whisper import WhisperTokenizer

tokenizer = WhisperTokenizer(language="vi")

sample_texts = ["Xin chào thế giới!"]
for text in sample_texts:
    ids = tokenizer.encode(text)
    print(f"Text: {text}")
    print(f"Token IDs: {ids}")
    decoded_text = tokenizer.batch_decode([ids])[0]
    print(f"Decoded Text: {decoded_text}")

## Dataset

### Grid

In [ ]:
import os
import glob
import json
import cv2

# Config
DATA_ROOT = "test_data" 
OUTPUT_MANIFEST = "manifest.jsonl"

# Find all .mpg video files 
search_pattern = os.path.join(DATA_ROOT, "s1_processed", "*.mpg")
video_files = glob.glob(search_pattern)

print(f"Found {len(video_files)} video files.")

entries = []
for video_path in video_files:
    
    # Find align file
    parent_dir = os.path.dirname(video_path) # test_data/s1_processed
    filename = os.path.basename(video_path)  # bbaf2n.mpg
    
    # Create path to align file: parent_dir -> align -> filename.align
    align_filename = filename.replace(".mpg", ".align")
    align_path = os.path.join(parent_dir, "align", align_filename)
    
    # Get Transcript
    transcript = ""
    if os.path.exists(align_path):
        try:
            with open(align_path, 'r') as f:
                words = [line.split()[2] for line in f if len(line.split()) >= 3 and line.split()[2] not in ['sil', 'sp']]
                transcript = " ".join(words)
        except Exception as e:
            print(f"Error reading align file {align_path}: {e}")
    else:
        # If still not found, print the path for debugging
        if len(entries) < 1: 
            print(f"Align file not found at: {align_path}")
    # Get Duration
    duration = 0.0
    try:
        cap = cv2.VideoCapture(video_path)
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            cnt = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            if fps > 0: duration = cnt / fps
        cap.release()
    except: pass

    # Create entry
    rel_path = os.path.relpath(video_path, DATA_ROOT)
    entries.append({
        "rel_path": rel_path,
        "text": transcript,
        "duration": round(duration, 2)
    })

# Write to manifest file
with open(OUTPUT_MANIFEST, 'w') as f:
    for e in entries:
        f.write(json.dumps(e) + "\n")

print(f" Created manifest: {OUTPUT_MANIFEST}")
# Print first few lines to verify transcripts are not empty
!head -n 2 {OUTPUT_MANIFEST}

## Data Loader

In [ ]:
!pip install openai-whisper

In [ ]:
from src.data.loader import build_dataloader
from src.data.tokenizers.whisper import WhisperTokenizer

test_config = {
    "data_root": "test_data",
    "train_manifest": "manifest.jsonl",
    "val_manifest": "manifest.jsonl",
    "batch_size": 2,
    "num_workers": 0,
    "use_precomputed_features": False,
}


tokenizer = WhisperTokenizer(language="en")
train_loader = build_dataloader(test_config, tokenizer, mode='train')


print(f"Getting one batch from train_loader...")
batch = next(iter(train_loader))

audio = batch['audio']  # (B, T)
print(f"Audio batch shape: {audio.shape}")

visual = batch['visual']  # (B, C, H, W, T)
print(f"Visual batch shape: {visual.shape}")

targets = batch['target']  # (B, S)
print(f"Targets batch shape: {targets.shape}")

# Check labels
for i in range(targets.size(0)):
    # Only row token, remove padding -100 
    token_row = targets[i]
    token_ids = token_row[token_row != -100].tolist()

    # decode to text
    decoded_text = tokenizer.batch_decode([token_ids])[0]
    orginal_text = batch['text'][i]

    print(f"Sample {i}:")
    print(f"Raw tokens: {token_row.tolist()[:10]}")
    print(f"Original Text: {orginal_text}")
    print(f"Decoded Text: {decoded_text}")

## Preprocessor